[rockyou.txt 다운로드(누르면 다운로드가 시작됩니다.)](https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt)

## 데이터 불러오기

In [13]:
filepath = "rockyou.txt"

passwords = []
with open(filepath, "r", encoding="latin-1") as f:
    for line in f:
        pw = line.strip()
        if pw:  # 빈 줄 무시
            passwords.append(pw)
print(f"총 비밀번호 개수: {len(passwords)}")

총 비밀번호 개수: 14344380


### 중복제거

In [14]:
passwords = set()
with open(filepath, "r", encoding="latin-1") as f:
    for line in f:
        pw = line.strip()
        if pw: passwords.add(pw)
        
passwords = list(passwords)
print(f"중복 제거 후 비밀번호 개수 :{len(passwords)}")

중복 제거 후 비밀번호 개수 :14343755


In [15]:
max_len = max(len(pw) for pw in passwords)
print(f"가장 긴 비밀번호 길이: {max_len}")

가장 긴 비밀번호 길이: 285


### 3자리 이하 ~ 40자리 이상 비밀번호 삭제

In [16]:
passwords_3 = [pw for pw in passwords if len(pw) >= 3]
passwords_40 = [pw for pw in passwords_3 if len(pw) <= 40]
passwords = passwords_40
print(f"길이 조정된 비밀번호 개수: {len(passwords_40)}")
print(f"총 비밀번호 개수: {len(passwords)}")

길이 조정된 비밀번호 개수: 14341812
총 비밀번호 개수: 14341812


### 공백 제거

In [17]:
no_space_passwords = [pw for pw in passwords if ' ' not in pw]
passwords = no_space_passwords
print(f"공백이 없는 비밀번호 개수: {len(no_space_passwords)}")
print(f"총 비밀번호 개수: {len(passwords)}")

공백이 없는 비밀번호 개수: 14272834
총 비밀번호 개수: 14272834


### ASCII외 인코딩 제거

In [18]:
import string
allowed_chars = set(string.ascii_letters + string.digits + "!@#$%^&*?/\\")  # 공백 제외 가능
sampled_passwords = [pw for pw in passwords if all(c in allowed_chars for c in pw)]
passwords = sampled_passwords
print(f"ASCII외 문자열 제외한 비밀번호 개수: {len(sampled_passwords)}")
print(f"총 비밀번호 개수: {len(passwords)}")

ASCII외 문자열 제외한 비밀번호 개수: 13688990
총 비밀번호 개수: 13688990


## 점수 라벨링

In [19]:
import pandas as pd
import re

def analyze_password(pw):
    length = len(pw)
    hasUpper = bool(re.search(r"[A-Z]", pw))
    hasLower = bool(re.search(r"[a-z]", pw))
    hasDigit = bool(re.search(r"[0-9]", pw))
    hasSpecial = bool(re.search(r"[!@#$%^&*?/\\]", pw))
    return {
        "password": pw,
        "length": length,
        "hasUpper": int(hasUpper),
        "hasLower": int(hasLower),
        "hasDigit": int(hasDigit),
        "hasSpecial": int(hasSpecial)
    }

def length_score(l):
    if l <= 9: return 0
    elif l <= 13: return 1
    elif l <= 17: return 2
    elif l <= 20: return 3
    elif l <= 25: return 4
    elif l <= 32: return 5
    else: return 6

def strength_label(totalScore):
    if totalScore <= 2: return "Weak"
    elif totalScore <= 4: return "Normal"
    else: return "Strong"


In [20]:
data = [analyze_password(pw) for pw in passwords]
df = pd.DataFrame(data)

df["length_score"] = df["length"].apply(length_score)
df["feature_score"] = df["hasUpper"] + df["hasLower"] + df["hasDigit"] + df["hasSpecial"]
df["total_score"] = df["length_score"] + df["feature_score"]
df["label"] = df["total_score"].apply(strength_label)

In [21]:
print(df[['total_score', 'label']].head(20))  # 점수랑 라벨 같이 찍어보기
print(df["label"].value_counts())

    total_score   label
0             2    Weak
1             2    Weak
2             1    Weak
3             3  Normal
4             2    Weak
5             2    Weak
6             3  Normal
7             3  Normal
8             2    Weak
9             1    Weak
10            2    Weak
11            1    Weak
12            1    Weak
13            2    Weak
14            3  Normal
15            1    Weak
16            2    Weak
17            1    Weak
18            2    Weak
19            2    Weak
label
Weak      10898061
Normal     2713988
Strong       76941
Name: count, dtype: int64


In [22]:
labelMap = {"Weak": "weak", "Normal": "normal", "Strong": "strong"}
df["label"] = df["label"].map(labelMap)

df = df[df["label"].isin(["weak", "normal", "strong"])]

min_count = df["label"].value_counts().min()

balanceDF = (
    df.groupby("label", group_keys=False)  # group_keys=False 추가
    .apply(lambda x: x.sample(n=min_count, random_state=42))
    .reset_index(drop=True)
)

print(balanceDF["label"].value_counts())

label
normal    76941
strong    76941
weak      76941
Name: count, dtype: int64


/var/folders/nd/5sbd_s6d58qct2dnqyxjhsxr0000gn/T/ipykernel_8246/2395946220.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=42))


In [23]:
balanceDF.to_csv("balanced_passwords_re.csv", index=False)